# Rebrickable - Ingest
_Created by Peer Grønnerup, April 22, 2026._

This notebook loads all CSV files from the [Rebrickable Downloads](https://rebrickable.com/downloads/) page into separate Delta tables in a Fabric Lakehouse.
The Rebrickable database contains a comprehensive catalog of LEGO parts, sets, colors, themes, and inventories. The CSV files are updated daily and hosted as `.csv.gz` archives.  
  
**This notebook performs the following tasks:**
- Downloads all 12 CSV files from Rebrickable (compressed as `.csv.gz`)
- Loads each file into a Spark DataFrame
- Writes each DataFrame as a Delta table with **full overwrite** and **schema merge** to handle drift

**Tables created:**
| Table | Description |
|---|---|
| `colors` | LEGO colors |
| `themes` | Set themes (e.g., City, Creator, Ninjago) |
| `part_categories` | Part category groupings |
| `parts` | LEGO parts catalog |
| `part_relationships` | Relationships between parts (print, mold, alternate, etc.) |
| `elements` | Element IDs linking parts to colors |
| `sets` | Official LEGO sets |
| `minifigs` | LEGO minifigures |
| `inventories` | Inventory records linking to sets/minifigs |
| `inventory_parts` | Parts contained in each inventory |
| `inventory_sets` | Sets contained in each inventory |
| `inventory_minifigs` | Minifigures contained in each inventory |


## Parameters
Configure the target Landing Lakehouse name and workspace ID below. These values can be overridden when calling the notebook from a pipeline.

In [ ]:
landing_lakehouse_name = "Landing"
landing_workspace_id = notebookutils.runtime.context.get("currentWorkspaceId")

## Configuration
Define the base download URL and the list of CSV files available from Rebrickable.

In [ ]:
REBRICKABLE_BASE_URL = "https://cdn.rebrickable.com/media/downloads"

REBRICKABLE_FILES = [
    "colors",
    "themes",
    "part_categories",
    "parts",
    "part_relationships",
    "elements",
    "sets",
    "minifigs",
    "inventories",
    "inventory_parts",
    "inventory_sets",
    "inventory_minifigs",
]

## Resolve Landing Lakehouse
Use Semantic Link to resolve the Landing Lakehouse based on the provided workspace ID and Lakehouse name.

In [ ]:
import sempy.fabric as fabric

if landing_workspace_id:
    lakehouse = fabric.resolve_item_id(
        item_name=landing_lakehouse_name,
        type="Lakehouse",
        workspace=landing_workspace_id,
    )
    lakehouse_path = f"abfss://{landing_workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse}/Tables"
else:
    # Fall back to the default attached Lakehouse
    lakehouse_path = "Tables"

print(f"Landing Lakehouse path: {lakehouse_path}")

## Download and Load Functions
Helper function to download a `.csv.gz` file from Rebrickable and return it as a Spark DataFrame.

In [ ]:
import requests
import gzip
import io
import pandas as pd

def download_and_load_csv(file_name: str) -> "DataFrame":
    """
    Downloads a .csv.gz file from Rebrickable and loads it into a Spark DataFrame.
    """
    url = f"{REBRICKABLE_BASE_URL}/{file_name}.csv.gz"
    print(f"Downloading: {url}")

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    decompressed = gzip.decompress(response.content)
    csv_text = decompressed.decode("utf-8")

    pdf = pd.read_csv(io.StringIO(csv_text))
    df = spark.createDataFrame(pdf)

    print(f"  Loaded {file_name}: {df.count()} rows, {len(df.columns)} columns")
    return df

## Ingest All Rebrickable Tables
Download each CSV file and write it as a Delta table to the Landing Lakehouse. Uses **overwrite** mode with **overwriteSchema** to handle any schema drift between loads.

In [ ]:
for file_name in REBRICKABLE_FILES:
    try:
        df = download_and_load_csv(file_name)

        table_path = f"{lakehouse_path}/{file_name}"

        df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .save(table_path)

        print(f"  Written to Delta table: {file_name}")

    except Exception as e:
        print(f"  ERROR loading {file_name}: {e}")
        raise

print("\nAll Rebrickable tables ingested successfully.")

## Verify Loaded Tables
Quick validation to confirm all tables were written and display row counts.

In [ ]:
print("Table verification:")
print("-" * 40)

for file_name in REBRICKABLE_FILES:
    try:
        table_path = f"{lakehouse_path}/{file_name}"
        df = spark.read.format("delta").load(table_path)
        print(f"  {file_name:<25} {df.count():>8} rows")
    except Exception as e:
        print(f"  {file_name:<25} ERROR: {e}")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = SparkSession.builder.getOrCreate()

schema = StructType([
    StructField("member_number", IntegerType(), False),
    StructField("name", StringType(), False)
])

data = [
    (1001, "Emmet Brickowski"),
    (1002, "Wyldstyle Lucy"),
    (1003, "Benny Spaceman"),
    (1004, "Lord Business"),
    (1005, "Kai Firefang"),
    (1006, "Jay Walker"),
    (1007, "Lloyd Garmadon"),
    (1008, "Nya Waterfall"),
    (1009, "Cole Brookstone"),
    (1010, "Zane Julien"),
    (1011, "Sensei Wu"),
    (1012, "Laval Lionheart"),
    (1013, "Eris Eagleton"),
    (1014, "Chase McCain"),
    (1015, "Rex Dangervest"),
    (1016, "Unikitty Cloudcuckoo"),
    (1017, "Metalbeard Pirate"),
    (1018, "Clutch Powers"),
]

df = spark.createDataFrame(data, schema)

table_path = f"{lakehouse_path}/members"

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(table_path)

In [ ]:
schema = StructType([
    StructField("member_number", IntegerType(), False),
    StructField("set_num", StringType(), False)
])

data = [
    # Emmet Brickowski - huge collector (20 sets, loves Star Wars & big builds)
    (1001, "75192-1"),  # UCS Millennium Falcon
    (1001, "75290-1"),  # Mos Eisley Cantina
    (1001, "10294-1"),  # Titanic
    (1001, "71043-1"),  # Hogwarts Castle
    (1001, "10276-1"),  # Colosseum
    (1001, "75313-1"),  # AT-AT
    (1001, "21054-1"),  # White House
    (1001, "42115-1"),  # Lamborghini Sián
    (1001, "10297-1"),  # Boutique Hotel
    (1001, "75309-1"),  # Republic Gunship
    (1001, "10300-1"),  # Back to the Future Time Machine
    (1001, "21330-1"),  # Home Alone
    (1001, "10305-1"),  # Lion Knights' Castle
    (1001, "75341-1"),  # Luke's Landspeeder
    (1001, "42143-1"),  # Ferrari Daytona SP3
    (1001, "10312-1"),  # Jazz Club
    (1001, "75375-1"),  # Millennium Falcon
    (1001, "10321-1"),  # Corvette
    (1001, "21336-1"),  # The Office
    (1001, "10307-1"),  # Eiffel Tower

    # Wyldstyle Lucy - solid collector (15 sets, mix of themes)
    (1002, "10255-1"),  # Assembly Square
    (1002, "21318-1"),  # Tree House
    (1002, "10270-1"),  # Bookshop
    (1002, "75257-1"),  # Millennium Falcon
    (1002, "10281-1"),  # Bonsai Tree
    (1002, "10280-1"),  # Flower Bouquet
    (1002, "21327-1"),  # Typewriter
    (1002, "10311-1"),  # Orchid
    (1002, "40460-1"),  # Roses
    (1002, "10278-1"),  # Police Station
    (1002, "10264-1"),  # Corner Garage
    (1002, "21325-1"),  # Medieval Blacksmith
    (1002, "10290-1"),  # Pickup Truck
    (1002, "10274-1"),  # Ghostbusters ECTO-1
    (1002, "76218-1"),  # Sanctum Sanctorum

    # Benny Spaceman - space fanatic (12 sets, mostly space/sci-fi)
    (1003, "10266-1"),  # NASA Apollo 11
    (1003, "21321-1"),  # ISS
    (1003, "10283-1"),  # NASA Discovery Shuttle
    (1003, "21309-1"),  # Saturn V
    (1003, "75355-1"),  # X-Wing
    (1003, "75367-1"),  # Venator-Class Republic Attack Cruiser
    (1003, "10497-1"),  # Galaxy Explorer
    (1003, "31117-1"),  # Space Shuttle Adventure
    (1003, "60349-1"),  # Lunar Space Station
    (1003, "60351-1"),  # Rocket Launch Center
    (1003, "60350-1"),  # Lunar Research Base
    (1003, "10340-1"),  # LEGO Tales of the Space Age

    # Lord Business - only buys exclusive sets (5 sets)
    (1004, "10276-1"),  # Colosseum
    (1004, "75192-1"),  # UCS Millennium Falcon
    (1004, "10294-1"),  # Titanic
    (1004, "10307-1"),  # Eiffel Tower
    (1004, "71043-1"),  # Hogwarts Castle

    # Kai Firefang - Ninjago & action themes (10 sets)
    (1005, "71767-1"),  # Ninja Dojo Temple
    (1005, "71774-1"),  # Lloyd's Golden Ultra Dragon
    (1005, "71741-1"),  # Ninjago City Gardens
    (1005, "71799-1"),  # Ninjago City Markets
    (1005, "70751-1"),  # Temple of Airjitzu
    (1005, "76261-1"),  # Spider-Man Final Battle
    (1005, "76178-1"),  # Daily Bugle
    (1005, "76218-1"),  # Sanctum Sanctorum
    (1005, "42143-1"),  # Ferrari Daytona SP3
    (1005, "42151-1"),  # Bugatti Bolide

    # Jay Walker - casual collector (3 sets)
    (1006, "10281-1"),  # Bonsai Tree
    (1006, "21327-1"),  # Typewriter
    (1006, "40460-1"),  # Roses

    # Lloyd Garmadon - city & architecture fan (8 sets)
    (1007, "21054-1"),  # White House
    (1007, "21044-1"),  # Paris
    (1007, "21042-1"),  # Statue of Liberty
    (1007, "21034-1"),  # London
    (1007, "60380-1"),  # Downtown
    (1007, "60364-1"),  # Street Skate Park
    (1007, "60372-1"),  # Police Training Academy
    (1007, "10278-1"),  # Police Station

    # Nya Waterfall - Technic enthusiast (14 sets)
    (1008, "42115-1"),  # Lamborghini Sián
    (1008, "42143-1"),  # Ferrari Daytona SP3
    (1008, "42141-1"),  # McLaren Formula 1
    (1008, "42145-1"),  # Airbus H175
    (1008, "42096-1"),  # Porsche 911 RSR
    (1008, "42110-1"),  # Land Rover Defender
    (1008, "42083-1"),  # Bugatti Chiron
    (1008, "42056-1"),  # Porsche 911 GT3 RS
    (1008, "42151-1"),  # Bugatti Bolide
    (1008, "42154-1"),  # Ford GT 2022
    (1008, "42111-1"),  # Dom's Dodge Charger
    (1008, "42126-1"),  # Ford F-150 Raptor
    (1008, "42125-1"),  # Ferrari 488 GTE
    (1008, "42127-1"),  # The Batman – Batmobile

    # Cole Brookstone - just getting started (2 sets)
    (1009, "10281-1"),  # Bonsai Tree
    (1009, "31120-1"),  # Medieval Castle

    # Zane Julien - Harry Potter fan (11 sets)
    (1010, "71043-1"),  # Hogwarts Castle
    (1010, "76405-1"),  # Hogwarts Express Collectors Edition
    (1010, "75978-1"),  # Diagon Alley
    (1010, "76391-1"),  # Hogwarts Icons
    (1010, "76419-1"),  # Hogwarts Castle and Grounds
    (1010, "76389-1"),  # Chamber of Secrets
    (1010, "76388-1"),  # Hogsmeade Village Visit
    (1010, "76395-1"),  # Hogwarts First Flying Lesson
    (1010, "76386-1"),  # Polyjuice Potion Mistake
    (1010, "76387-1"),  # Fluffy Encounter
    (1010, "76402-1"),  # Dumbledore's Office

    # Laval Lionheart - classic collector (7 sets)
    (1011, "10305-1"),  # Lion Knights' Castle
    (1011, "10332-1"),  # Medieval Town Square
    (1011, "21325-1"),  # Medieval Blacksmith
    (1011, "31120-1"),  # Medieval Castle
    (1011, "10316-1"),  # Lord of the Rings: Rivendell
    (1011, "21348-1"),  # Dungeons & Dragons
    (1011, "10698-1"),  # Large Creative Brick Box

    # Eris Eagleton - picks a few favorites (4 sets)
    (1012, "10280-1"),  # Flower Bouquet
    (1012, "10311-1"),  # Orchid
    (1012, "10313-1"),  # Wildflower Bouquet
    (1012, "10289-1"),  # Bird of Paradise

    # Chase McCain - City superfan (16 sets)
    (1013, "60380-1"),  # Downtown
    (1013, "60364-1"),  # Street Skate Park
    (1013, "60372-1"),  # Police Training Academy
    (1013, "10278-1"),  # Police Station
    (1013, "60349-1"),  # Lunar Space Station
    (1013, "60337-1"),  # Express Passenger Train
    (1013, "60336-1"),  # Freight Train
    (1013, "60335-1"),  # Train Station
    (1013, "60327-1"),  # Horse Transporter
    (1013, "60324-1"),  # Mobile Crane
    (1013, "60316-1"),  # Police Station
    (1013, "60315-1"),  # Police Mobile Command Truck
    (1013, "60292-1"),  # Town Center
    (1013, "60271-1"),  # Main Square
    (1013, "60197-1"),  # Passenger Train
    (1013, "60198-1"),  # Cargo Train

    # Rex Dangervest - random fun picks (6 sets)
    (1014, "10300-1"),  # Back to the Future Time Machine
    (1014, "21336-1"),  # The Office
    (1014, "21330-1"),  # Home Alone
    (1014, "10274-1"),  # Ghostbusters ECTO-1
    (1014, "21319-1"),  # Friends Central Perk
    (1014, "21328-1"),  # Seinfeld

    # Unikitty Cloudcuckoo - colorful & creative sets (9 sets)
    (1015, "10280-1"),  # Flower Bouquet
    (1015, "10281-1"),  # Bonsai Tree
    (1015, "10311-1"),  # Orchid
    (1015, "10313-1"),  # Wildflower Bouquet
    (1015, "41703-1"),  # Friendship Tree House
    (1015, "43222-1"),  # Disney Castle
    (1015, "43197-1"),  # Ice Castle
    (1015, "10698-1"),  # Large Creative Brick Box
    (1015, "11717-1"),  # Bricks Bricks Plates

    # Metalbeard Pirate - pirate & adventure (5 sets)
    (1016, "21322-1"),  # Pirates of Barracuda Bay
    (1016, "10320-1"),  # Eldorado Fortress
    (1016, "31109-1"),  # Pirate Ship
    (1016, "21316-1"),  # The Flintstones
    (1016, "10305-1"),  # Lion Knights' Castle

    # Clutch Powers - completionist (18 sets, wide range)
    (1017, "75192-1"),  # UCS Millennium Falcon
    (1017, "10276-1"),  # Colosseum
    (1017, "10294-1"),  # Titanic
    (1017, "42115-1"),  # Lamborghini Sián
    (1017, "71043-1"),  # Hogwarts Castle
    (1017, "10307-1"),  # Eiffel Tower
    (1017, "21054-1"),  # White House
    (1017, "21309-1"),  # Saturn V
    (1017, "10305-1"),  # Lion Knights' Castle
    (1017, "10255-1"),  # Assembly Square
    (1017, "21325-1"),  # Medieval Blacksmith
    (1017, "76178-1"),  # Daily Bugle
    (1017, "42143-1"),  # Ferrari Daytona SP3
    (1017, "10312-1"),  # Jazz Club
    (1017, "10300-1"),  # Back to the Future Time Machine
    (1017, "10316-1"),  # Lord of the Rings: Rivendell
    (1017, "43222-1"),  # Disney Castle
    (1017, "10283-1"),  # NASA Discovery Shuttle
]

df = spark.createDataFrame(data, schema)
table_path = f"{lakehouse_path}/member_sets"

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(table_path)